# 06. Evaluasi Model Komparatif dan Lapisan Keputusan Risiko Kapasitas

Notebook ini merupakan tahap integrasi sistem cerdas: membandingkan seluruh model secara menyeluruh, menetapkan model terbaik, dan menerjemahkan batas atas prediksi interval ketidakpastian (95%) menjadi **Sistem Peringatan Dini Risiko Kapasitas (*Capacity Risk Alert*)** bagi pemangku kebijakan KRL Jabodetabek.

### Komponen Utama:
1. **Rangkuman Evaluasi Model Komparatif**: Membandingkan akurasi titik (*MASE, RMSE, MAE*) dan interval (*PICP, MPIW, Winkler*).
2. **Lapisan Keputusan (*Decision Layer*)**: Mengklasifikasikan risiko bulan masa depan menjadi **Aman**, **Waspada**, atau **Kritis** berdasarkan batas kapasitas.
3. **Analisis Sensitivitas Skenario**: Menguji stabilitas status risiko jika ambang kapasitas bergeser $\pm 5\%$ dan $\pm 10\%$.
4. **Rekomendasi Manajerial & Operasional**: Arahan penambahan headway dan armada KAI Commuter.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../src"))

import config as cfg
import risk as rsk
import metrics as mt

FOLDER_OUTPUT = "../outputs"
FOLDER_GAMBAR = "../reports/figures"
os.makedirs(FOLDER_OUTPUT, exist_ok=True)
os.makedirs(FOLDER_GAMBAR, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Tabel Komparasi Akurasi Seluruh Model
Menilai performa 6 model yang telah diuji melalui rolling-origin backtest.

In [ ]:
met_komp = pd.read_csv(os.path.join(FOLDER_OUTPUT, 'metrik_komparasi_final.csv'))
sub_bersih = met_komp[met_komp['subset'] == 'bersih'].sort_values('MASE').reset_index(drop=True)

print("=== PERBANDINGAN PERFORMA SELURUH MODEL (PERIODE BERSIH PASCA-COVID) ===")
display(sub_bersih)

met_h = pd.read_csv(os.path.join(FOLDER_OUTPUT, 'metrik_per_horizon_final.csv'))
piv = met_h[met_h['subset'] == 'bersih'].pivot(index='horizon', columns='model', values='MASE')
print("\n=== MASE PER HORIZON (SUBSET BERSIH) ===")
display(piv[['seasonal_naive', 'naive', 'ETS_tanpa_musim', 'SARIMA', 'SARIMAX', 'LightGBM']].round(3))

## 2. Klasifikasi Risiko Kapasitas 6 Bulan ke Depan
Ambang kapasitas KRL ditetapkan sebesar **34.000 ribu orang/bulan** (sekitar 1,1 juta penumpang per hari kerja).
- **Aman**: Batas atas < 30.600 ribu (90% ambang).
- **Waspada**: Batas atas antara 30.600 - 34.000 ribu.
- **Kritis**: Batas atas $\ge$ 34.000 ribu.

In [ ]:
df_fut = pd.read_csv(os.path.join(FOLDER_OUTPUT, 'prediksi_future_6bln.csv'))
df_risiko = rsk.klasifikasi_tabel_risiko(df_fut, ambang_kapasitas=cfg.AMBANG_KAPASITAS, batas_waspada=cfg.BATAS_WASPADA)

df_risiko.to_csv(os.path.join(FOLDER_OUTPUT, 'tabel_status_risiko_kapasitas.csv'), index=False)
print("=== STATUS RISIKO KAPASITAS MASA DEPAN (SEPTEMBER 2026 - FEBRUARI 2027) ===")
display(df_risiko[['horizon', 'tanggal_target', 'y_pred_titik', 'batas_atas_95', 'okupansi_batas_atas_%', 'status_risiko']])

## 3. Visualisasi Peringatan Dini Risiko Kapasitas

In [ ]:
plt.figure(figsize=(12, 6))
tgl = pd.to_datetime(df_risiko['tanggal_target'])

plt.plot(tgl, df_risiko['y_pred_titik'], 'g--o', label='Prakiraan Titik Permintaan', linewidth=2)
plt.plot(tgl, df_risiko['batas_atas_95'], 'r-o', label='Batas Atas Ketidakpastian (95%)', linewidth=2.5)
plt.fill_between(tgl, df_risiko['batas_bawah_95'], df_risiko['batas_atas_95'], color='green', alpha=0.15, label='Interval Prediksi 95%')

plt.axhline(cfg.AMBANG_KAPASITAS, color='red', linestyle='-', linewidth=2, label=f'Ambang Kritis (100% = {cfg.AMBANG_KAPASITAS:,.0f} ribu)')
plt.axhline(cfg.AMBANG_KAPASITAS * cfg.BATAS_WASPADA, color='orange', linestyle='--', linewidth=1.5, label=f'Batas Waspada (90% = {cfg.AMBANG_KAPASITAS*cfg.BATAS_WASPADA:,.0f} ribu)')

# Beri anotasi status risiko
for i, row in df_risiko.iterrows():
    warna = 'red' if row['status_risiko'] == 'Kritis' else ('orange' if row['status_risiko'] == 'Waspada' else 'green')
    plt.annotate(f"{row['status_risiko']}\n({row['okupansi_batas_atas_%']}%%)",
                 (tgl.iloc[i], row['batas_atas_95'] + 300),
                 ha='center', fontsize=9, fontweight='bold', color=warna)

plt.title('Dashboard Peringatan Dini Risiko Kapasitas KRL Jabodetabek (6 Bulan ke Depan)', fontsize=13, fontweight='bold')
plt.xlabel('Bulan')
plt.ylabel('Penumpang (Ribu Orang)')
plt.ylim(25000, 38000)
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(FOLDER_GAMBAR, 'risiko_01_kapasitas_alert.png'), dpi=150)
plt.show()

## 4. Analisis Sensitivitas Ambang Kapasitas
Bagaimana status risiko berubah jika kapasitas efektif berkurang atau bertambah?

In [ ]:
df_sens = rsk.analisis_sensitivitas(df_fut, ambang_dasar=cfg.AMBANG_KAPASITAS)
df_sens.to_csv(os.path.join(FOLDER_OUTPUT, 'analisis_sensitivitas_kapasitas.csv'), index=False)
print("=== ANALISIS SENSITIVITAS RISIKO KAPASITAS ===")
display(df_sens)

## 5. Rekomendasi Manajerial & Mitigasi Operasional

1. **Bulan Kritis (Okupansi Batas Atas $\ge 100\%$)**:
   - Pada bulan-bulan di mana batas atas melampaui kapasitas 34 juta (misal horizon 2, 4, 5, dan 6), potensi stasiun padat dan penumpukan penumpang peron sangat tinggi.
   - **Tindakan**: Penambahan rangkaian trainset 12-SF pada lintas terpadat (Bogor Line & Cikarang Line), pemendekan headway jam sibuk (peak hour), dan pembatasan arus *tap-in* stasiun transit (Manggarai).
2. **Bulan Waspada (Okupansi Batas Atas $90\% - 100\%$)**:
   - Penumpang terkonsentrasi namun masih dalam batas kendali.
   - **Tindakan**: Pengalihan armada feeder dan optimalisasi integrasi moda (LRT/Transjakarta) untuk mengurai kepadatan pintu keluar stasiun utama.